In [ ]:
import os
import re
import multiprocessing as mp
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
import pysam
import matplotlib.pyplot as plt
import matplotlib as mpl

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **kwargs: x

try:
    from IPython.display import display
except ImportError:
    display = print


# ============================================================
# Settings
# ============================================================

bam_root = Path(
    "/home/913/dk4874/scratch/XCI_RA_aging/processed_data/"
    "singlecell/BAM/demuxed_chrX_dedup"
)

possible_gtfs = [
    Path(
        "/g/data/oo78/dk4874/genomes/cellranger/"
        "homo_sapiens/refdata-gex-GRCh38-2024-A/"
        "genes/genes.gtf"
    )
]

gtf_path = next(
    (path for path in possible_gtfs if path.exists()),
    None,
)

if gtf_path is None:
    raise FileNotFoundError(
        "None of the possible GTF paths exist. Set gtf_path manually."
    )

snp_path = Path(
    "/home/913/dk4874/scratch/primateXInactivation/"
    "scripts/paper/closest_het_snp_distances.csv"
)

outdir = Path("plots")
outdir.mkdir(parents=True, exist_ok=True)

# Use exactly these eight BAM donors.
samples = [
    "DRG_0198_01",
    "DRG_0199_01",
    "DRG_0341_01",
    "DRG_1409_01",
    "DRG_1215_02",
    "DRG_2324_02",
    "DRG_2902_02",
    "DRG_5467_02",
]

gn_tag_name = "GN"
min_mapq = 0
keep_secondary = False
keep_supplementary = False

# For each read × gene, retain the transcript with the
# greatest exonic overlap.
best_transcript_per_read_gene = True

# Set this to the number of CPUs requested in the PBS job.
n_processes = min(16, os.cpu_count() or 1)

exon_bin_size = 10_000

invalid_gene_tags = {
    "",
    "-",
    "NA",
    "None",
    "none",
    "nan",
    "NaN",
    "NULL",
    "null",
}


# ============================================================
# Plot settings
# ============================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",

    "font.size": 6.0,
    "axes.titlesize": 6.5,
    "axes.labelsize": 6.0,
    "xtick.labelsize": 5.5,
    "ytick.labelsize": 5.5,
    "legend.fontsize": 5.5,

    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 2.0,
    "ytick.major.size": 2.0,
})

MM_TO_INCH = 1 / 25.4
FIGSIZE_50MM = (
    50 * MM_TO_INCH,
    50 * MM_TO_INCH,
)


# ============================================================
# GTF parsing
# ============================================================

def parse_gtf_attrs(attr_string):
    attrs = {}

    for item in attr_string.strip().split(";"):
        item = item.strip()

        if not item:
            continue

        if " " in item:
            key, value = item.split(" ", 1)
            attrs[key] = value.strip().strip('"')

    return attrs


def is_chrX(chrom):
    return chrom in {"chrX", "X", "23"}


def merge_intervals(intervals):
    if not intervals:
        return []

    intervals = sorted(intervals)
    merged = [list(intervals[0])]

    for start, end in intervals[1:]:
        last_start, last_end = merged[-1]

        if start <= last_end:
            merged[-1][1] = max(last_end, end)
        else:
            merged.append([start, end])

    return [
        (start, end)
        for start, end in merged
    ]


def parse_gtf_chrX_transcripts(gtf_path):
    tx_raw = defaultdict(
        lambda: {
            "gene": None,
            "gene_id": None,
            "transcript": None,
            "chrom": None,
            "strand": None,
            "exons": [],
        }
    )

    with open(gtf_path, "r") as handle:
        for line in handle:
            if line.startswith("#"):
                continue

            parts = line.rstrip("\n").split("\t")

            if len(parts) < 9:
                continue

            (
                chrom,
                source,
                feature,
                start,
                end,
                score,
                strand,
                frame,
                attrs_raw,
            ) = parts

            if feature != "exon":
                continue

            if not is_chrX(chrom):
                continue

            attrs = parse_gtf_attrs(attrs_raw)

            gene_id = attrs.get("gene_id")
            gene_name = attrs.get("gene_name", gene_id)
            transcript_id = attrs.get("transcript_id")

            if gene_id is None or transcript_id is None:
                continue

            # Convert GTF 1-based inclusive coordinates to
            # Python/BAM 0-based half-open coordinates.
            start0 = int(start) - 1
            end0 = int(end)

            rec = tx_raw[transcript_id]

            rec["gene"] = gene_name
            rec["gene_id"] = gene_id
            rec["transcript"] = transcript_id
            rec["chrom"] = chrom
            rec["strand"] = strand
            rec["exons"].append((start0, end0))

    transcript_models = []

    for transcript_id, rec in tx_raw.items():
        exons = merge_intervals(rec["exons"])

        if not exons:
            continue

        strand = rec["strand"]

        if strand == "+":
            exons_5p_to_3p = sorted(exons)
        elif strand == "-":
            exons_5p_to_3p = sorted(
                exons,
                reverse=True,
            )
        else:
            continue

        tx_exons = []
        cumulative_position = 0

        for exon_start, exon_end in exons_5p_to_3p:
            exon_length = exon_end - exon_start

            tx_exons.append({
                "chrom": rec["chrom"],
                "start": exon_start,
                "end": exon_end,
                "strand": strand,
                "gene": rec["gene"],
                "gene_id": rec["gene_id"],
                "transcript": transcript_id,
                "tx_exonic_start": cumulative_position,
                "tx_exonic_end": (
                    cumulative_position + exon_length
                ),
                "tx_spliced_len": None,
            })

            cumulative_position += exon_length

        for exon in tx_exons:
            exon["tx_spliced_len"] = cumulative_position

        transcript_models.append({
            "gene": rec["gene"],
            "gene_id": rec["gene_id"],
            "transcript": transcript_id,
            "chrom": rec["chrom"],
            "strand": strand,
            "tx_spliced_len": cumulative_position,
            "exons": tx_exons,
        })

    return transcript_models


# ============================================================
# BAM helpers
# ============================================================

def pick_chrX_from_bam(bam):
    references = set(bam.references)

    for chromosome in ["chrX", "X", "23"]:
        if chromosome in references:
            return chromosome

    raise ValueError(
        "Could not find chrX, X or 23 in BAM references. "
        f"First references: {list(bam.references)[:20]}"
    )


def get_gn_values(read, tag_name="GN"):
    try:
        gn = read.get_tag(tag_name)
    except KeyError:
        return set()

    if gn is None:
        return set()

    gn = str(gn).strip()

    if gn in invalid_gene_tags:
        return set()

    values = re.split(r"[;,|]", gn)

    return {
        value.strip()
        for value in values
        if value.strip() not in invalid_gene_tags
    }


def exon_matches_gn(exon, gn_values):
    return (
        str(exon["gene"]) in gn_values
        or str(exon["gene_id"]) in gn_values
    )


def build_exon_bin_index(
    transcript_models,
    bam_chrX,
    bin_size=10_000,
):
    bins = defaultdict(list)
    exon_id = 0

    for model in transcript_models:
        for exon in model["exons"]:
            exon = dict(exon)
            exon["chrom"] = bam_chrX
            exon["exon_id"] = exon_id

            exon_id += 1

            first_bin = exon["start"] // bin_size
            last_bin = (exon["end"] - 1) // bin_size

            for bin_number in range(
                first_bin,
                last_bin + 1,
            ):
                bins[bin_number].append(exon)

    return bins


def query_exon_bins(
    exon_bins,
    block_start,
    block_end,
    bin_size=10_000,
):
    first_bin = block_start // bin_size
    last_bin = (block_end - 1) // bin_size

    seen = set()
    hits = []

    for bin_number in range(
        first_bin,
        last_bin + 1,
    ):
        for exon in exon_bins.get(bin_number, []):
            exon_id = exon["exon_id"]

            if exon_id in seen:
                continue

            seen.add(exon_id)

            if (
                exon["start"] < block_end
                and exon["end"] > block_start
            ):
                hits.append(exon)

    return hits


def genomic_overlap_to_tx_coords(
    exon,
    overlap_start,
    overlap_end,
):
    exon_start = exon["start"]
    exon_end = exon["end"]
    strand = exon["strand"]
    tx_exonic_start = exon["tx_exonic_start"]

    if strand == "+":
        tx_start = (
            tx_exonic_start
            + overlap_start
            - exon_start
        )

        tx_end = (
            tx_exonic_start
            + overlap_end
            - exon_start
        )

    else:
        tx_start = (
            tx_exonic_start
            + exon_end
            - overlap_end
        )

        tx_end = (
            tx_exonic_start
            + exon_end
            - overlap_start
        )

    return tx_start, tx_end


# ============================================================
# Per-sample worker
# ============================================================

def process_one_sample(args):
    (
        sample,
        bam_path,
        transcript_models,
        gn_tag_name,
        min_mapq,
        keep_secondary,
        keep_supplementary,
        best_transcript_per_read_gene,
        exon_bin_size,
    ) = args

    bam_path = Path(bam_path)

    try:
        bam_exists = bam_path.exists()
    except OSError as error:
        return {
            "sample": sample,
            "rows": [],
            "summary": {
                "sample": sample,
                "status": f"bam_stat_error: {error}",
                "n_alignments_seen": 0,
                "n_alignments_used": 0,
                "n_valid_gn": 0,
                "n_gn_matched_exon_overlap": 0,
                "n_rows": 0,
            },
        }

    if not bam_exists:
        return {
            "sample": sample,
            "rows": [],
            "summary": {
                "sample": sample,
                "status": "missing_bam",
                "n_alignments_seen": 0,
                "n_alignments_used": 0,
                "n_valid_gn": 0,
                "n_gn_matched_exon_overlap": 0,
                "n_rows": 0,
            },
        }

    rows = []

    n_alignments_seen = 0
    n_alignments_used = 0
    n_valid_gn = 0
    n_gn_matched_exon_overlap = 0

    with pysam.AlignmentFile(
        str(bam_path),
        "rb",
    ) as bam:
        bam_chrX = pick_chrX_from_bam(bam)

        exon_bins = build_exon_bin_index(
            transcript_models,
            bam_chrX=bam_chrX,
            bin_size=exon_bin_size,
        )

        try:
            read_iter = bam.fetch(bam_chrX)
        except ValueError:
            read_iter = bam

        for read in read_iter:
            n_alignments_seen += 1

            if read.is_unmapped:
                continue

            if read.reference_id < 0:
                continue

            if (
                bam.get_reference_name(read.reference_id)
                != bam_chrX
            ):
                continue

            if read.mapping_quality < min_mapq:
                continue

            if (
                read.is_secondary
                and not keep_secondary
            ):
                continue

            if (
                read.is_supplementary
                and not keep_supplementary
            ):
                continue

            n_alignments_used += 1

            gn_values = get_gn_values(
                read,
                tag_name=gn_tag_name,
            )

            if not gn_values:
                continue

            n_valid_gn += 1

            blocks = read.get_blocks()

            if not blocks:
                continue

            tx_hits = {}

            for block_start, block_end in blocks:
                overlapping_exons = query_exon_bins(
                    exon_bins,
                    block_start,
                    block_end,
                    bin_size=exon_bin_size,
                )

                for exon in overlapping_exons:
                    if not exon_matches_gn(
                        exon,
                        gn_values,
                    ):
                        continue

                    overlap_start = max(
                        block_start,
                        exon["start"],
                    )

                    overlap_end = min(
                        block_end,
                        exon["end"],
                    )

                    if overlap_end <= overlap_start:
                        continue

                    tx_start, tx_end = (
                        genomic_overlap_to_tx_coords(
                            exon,
                            overlap_start,
                            overlap_end,
                        )
                    )

                    key = (
                        sample,
                        read.query_name,
                        exon["gene"],
                        exon["gene_id"],
                        exon["transcript"],
                    )

                    if key not in tx_hits:
                        tx_hits[key] = {
                            "sample": sample,
                            "gene": exon["gene"],
                            "gene_id": exon["gene_id"],
                            "GN_tag": ";".join(
                                sorted(gn_values)
                            ),
                            "transcript": exon["transcript"],
                            "read_id": read.query_name,
                            "tx_spliced_len": exon[
                                "tx_spliced_len"
                            ],
                            "tx_start_bp": tx_start,
                            "tx_end_bp": tx_end,
                            "read_exonic_aligned_bp": 0,
                        }

                    rec = tx_hits[key]

                    rec["tx_start_bp"] = min(
                        rec["tx_start_bp"],
                        tx_start,
                    )

                    rec["tx_end_bp"] = max(
                        rec["tx_end_bp"],
                        tx_end,
                    )

                    rec["read_exonic_aligned_bp"] += (
                        overlap_end - overlap_start
                    )

            if not tx_hits:
                continue

            n_gn_matched_exon_overlap += 1

            hit_rows = list(tx_hits.values())

            if best_transcript_per_read_gene:
                best = {}

                for rec in hit_rows:
                    key = (
                        rec["sample"],
                        rec["read_id"],
                        rec["gene"],
                    )

                    if key not in best:
                        best[key] = rec
                        continue

                    old = best[key]

                    better_overlap = (
                        rec["read_exonic_aligned_bp"]
                        > old["read_exonic_aligned_bp"]
                    )

                    tie_shorter_transcript = (
                        rec["read_exonic_aligned_bp"]
                        == old["read_exonic_aligned_bp"]
                        and rec["tx_spliced_len"]
                        < old["tx_spliced_len"]
                    )

                    if (
                        better_overlap
                        or tie_shorter_transcript
                    ):
                        best[key] = rec

                hit_rows = list(best.values())

            for rec in hit_rows:
                transcript_length = rec[
                    "tx_spliced_len"
                ]

                if transcript_length <= 0:
                    continue

                relative_start = (
                    rec["tx_start_bp"]
                    / transcript_length
                )

                relative_end = (
                    rec["tx_end_bp"]
                    / transcript_length
                )

                relative_start = min(
                    max(relative_start, 0),
                    1,
                )

                relative_end = min(
                    max(relative_end, 0),
                    1,
                )

                if relative_end < relative_start:
                    relative_start, relative_end = (
                        relative_end,
                        relative_start,
                    )

                rows.append({
                    "sample": rec["sample"],
                    "gene": rec["gene"],
                    "gene_id": rec["gene_id"],
                    "GN_tag": rec["GN_tag"],
                    "transcript": rec["transcript"],
                    "read_id": rec["read_id"],
                    "rel_start_exonic": relative_start,
                    "rel_end_exonic": relative_end,
                    "tx_spliced_len": transcript_length,
                    "tx_start_bp": rec["tx_start_bp"],
                    "tx_end_bp": rec["tx_end_bp"],
                    "read_exonic_aligned_bp": rec[
                        "read_exonic_aligned_bp"
                    ],
                })

    return {
        "sample": sample,
        "rows": rows,
        "summary": {
            "sample": sample,
            "status": "ok",
            "n_alignments_seen": n_alignments_seen,
            "n_alignments_used": n_alignments_used,
            "n_valid_gn": n_valid_gn,
            "n_gn_matched_exon_overlap": (
                n_gn_matched_exon_overlap
            ),
            "n_rows": len(rows),
        },
    }


# ============================================================
# Parse GTF and run BAM processing
# ============================================================

print(f"[info] Using GTF: {gtf_path}")

transcript_models = parse_gtf_chrX_transcripts(
    gtf_path
)

print(
    f"[info] Parsed "
    f"{len(transcript_models):,} "
    "chrX transcript models"
)

print(f"[info] Samples: {len(samples)}")
print(samples)

tasks = []

for sample in samples:
    bam_path = (
        bam_root
        / sample
        / f"{sample}_chrX_deduped.bam"
    )

    tasks.append((
        sample,
        str(bam_path),
        transcript_models,
        gn_tag_name,
        min_mapq,
        keep_secondary,
        keep_supplementary,
        best_transcript_per_read_gene,
        exon_bin_size,
    ))

all_rows = []
summary_rows = []

if n_processes <= 1:
    for task in tqdm(
        tasks,
        desc="Processing samples",
    ):
        result = process_one_sample(task)

        all_rows.extend(result["rows"])
        summary_rows.append(result["summary"])

        print(result["summary"])

else:
    ctx = mp.get_context("fork")

    with ProcessPoolExecutor(
        max_workers=n_processes,
        mp_context=ctx,
    ) as executor:
        futures = [
            executor.submit(
                process_one_sample,
                task,
            )
            for task in tasks
        ]

        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc="Processing samples",
        ):
            result = future.result()

            all_rows.extend(result["rows"])
            summary_rows.append(result["summary"])

            print(result["summary"])

read_tx_exonic = pd.DataFrame(all_rows)
processing_summary = pd.DataFrame(summary_rows)

print(
    f"[info] read_tx_exonic shape: "
    f"{read_tx_exonic.shape}"
)

display(
    processing_summary
    .sort_values("sample")
    .reset_index(drop=True)
)


# ============================================================
# Exact gene-reach calculation
# ============================================================

if read_tx_exonic.empty:
    raise ValueError(
        "read_tx_exonic is empty. Check BAM paths, GN tags, "
        "GTF gene names/IDs, chrX naming and MAPQ filters."
    )

read_tx_exonic["read_end_bp_from_5p"] = (
    read_tx_exonic["rel_end_exonic"]
    * read_tx_exonic["tx_spliced_len"]
)

read_tx_exonic["percent_TR_covered"] = (
    100
    * (
        read_tx_exonic["rel_end_exonic"]
        - read_tx_exonic["rel_start_exonic"]
    )
).clip(0, 100)

gene_reach_chrX = (
    read_tx_exonic
    .groupby("gene")
    .agg(
        n_reads=(
            "read_id",
            "count",
        ),
        median_reach_bp=(
            "read_end_bp_from_5p",
            "median",
        ),
        mean_reach_bp=(
            "read_end_bp_from_5p",
            "mean",
        ),
        p25_reach_bp=(
            "read_end_bp_from_5p",
            lambda values: np.percentile(
                values,
                25,
            ),
        ),
        p75_reach_bp=(
            "read_end_bp_from_5p",
            lambda values: np.percentile(
                values,
                75,
            ),
        ),
        median_tx_len=(
            "tx_spliced_len",
            "median",
        ),
        median_percent_covered=(
            "percent_TR_covered",
            "median",
        ),
    )
    .reset_index()
)

display(
    gene_reach_chrX
    .sort_values(
        "n_reads",
        ascending=False,
    )
    .head(30)
)


# ============================================================
# Save BAM-derived reach tables
# ============================================================

read_tx_exonic_path = (
    outdir
    / "read_tx_exonic_GN_parallel.csv.gz"
)

gene_reach_path = (
    outdir
    / "gene_reach_chrX_GN_parallel.csv"
)

processing_summary_path = (
    outdir
    / "read_tx_exonic_GN_parallel_processing_summary.csv"
)

read_tx_exonic.to_csv(
    read_tx_exonic_path,
    index=False,
)

gene_reach_chrX.to_csv(
    gene_reach_path,
    index=False,
)

processing_summary.to_csv(
    processing_summary_path,
    index=False,
)

print(f"[done] Wrote {read_tx_exonic_path}")
print(f"[done] Wrote {gene_reach_path}")
print(f"[done] Wrote {processing_summary_path}")


# ============================================================
# Gene ID to symbol mapping
# ============================================================

gene_id_to_symbol = {}

with open(gtf_path) as handle:
    for line in handle:
        if line.startswith("#"):
            continue

        fields = line.rstrip("\n").split("\t")

        if len(fields) < 9:
            continue

        attributes = fields[8]

        gene_id_match = re.search(
            r'gene_id "([^"]+)"',
            attributes,
        )

        gene_name_match = re.search(
            r'gene_name "([^"]+)"',
            attributes,
        )

        if gene_id_match and gene_name_match:
            clean_gene_id = (
                gene_id_match
                .group(1)
                .split(".")[0]
            )

            gene_id_to_symbol[clean_gene_id] = (
                gene_name_match.group(1)
            )

print(
    f"[info] Loaded "
    f"{len(gene_id_to_symbol):,} "
    "gene ID-to-symbol mappings"
)


# ============================================================
# Load SNP-distance table
# ============================================================

snp_df = pd.read_csv(snp_path)

required_snp_columns = {
    "sample",
    "gene",
    "TSS_distance_RNA",
    "TES_distance_RNA",
}

missing_snp_columns = (
    required_snp_columns
    .difference(snp_df.columns)
)

if missing_snp_columns:
    raise ValueError(
        "The SNP-distance table is missing columns: "
        f"{sorted(missing_snp_columns)}"
    )

print(
    f"[info] Loaded SNP distances for "
    f"{snp_df['sample'].nunique():,} individuals"
)


# ============================================================
# TSS summary
# ============================================================

tss_individual = (
    snp_df
    .dropna(
        subset=["TSS_distance_RNA"]
    )
    .copy()
)

tss_individual["gene_id_clean"] = (
    tss_individual["gene"]
    .astype(str)
    .str.split(".")
    .str[0]
)

tss_individual["gene_symbol"] = (
    tss_individual["gene_id_clean"]
    .map(gene_id_to_symbol)
)

tss_individual["TSS_distance_RNA"] = (
    pd.to_numeric(
        tss_individual["TSS_distance_RNA"],
        errors="coerce",
    )
)

tss_individual = tss_individual.dropna(
    subset=[
        "sample",
        "gene_symbol",
        "TSS_distance_RNA",
    ]
)

tss_reach_vs_snp = gene_reach_chrX.merge(
    tss_individual,
    left_on="gene",
    right_on="gene_symbol",
    how="inner",
)

tss_reach_vs_snp[
    "reachable_by_illumina_150bp"
] = (
    tss_reach_vs_snp["TSS_distance_RNA"]
    <= 150
)

tss_reach_vs_snp[
    "reachable_by_nanopore_median"
] = (
    tss_reach_vs_snp["median_reach_bp"]
    >= tss_reach_vs_snp["TSS_distance_RNA"]
)

tss_reach_vs_snp[
    "reachable_by_nanopore_p75"
] = (
    tss_reach_vs_snp["p75_reach_bp"]
    >= tss_reach_vs_snp["TSS_distance_RNA"]
)

tss_individual_reach_summary = (
    tss_reach_vs_snp
    .groupby("sample")
    .agg(
        n_gene_tests=(
            "gene_symbol",
            "count",
        ),
        illumina_150bp_reachable=(
            "reachable_by_illumina_150bp",
            "sum",
        ),
        nanopore_median_reachable=(
            "reachable_by_nanopore_median",
            "sum",
        ),
        nanopore_p75_reachable=(
            "reachable_by_nanopore_p75",
            "sum",
        ),
    )
    .reset_index()
)

for method in [
    "illumina_150bp",
    "nanopore_median",
    "nanopore_p75",
]:
    tss_individual_reach_summary[
        f"{method}_fraction"
    ] = (
        tss_individual_reach_summary[
            f"{method}_reachable"
        ]
        / tss_individual_reach_summary[
            "n_gene_tests"
        ]
    )


# ============================================================
# TES summary
# ============================================================

tes_individual = (
    snp_df
    .dropna(
        subset=["TES_distance_RNA"]
    )
    .copy()
)

tes_individual["gene_id_clean"] = (
    tes_individual["gene"]
    .astype(str)
    .str.split(".")
    .str[0]
)

tes_individual["gene_symbol"] = (
    tes_individual["gene_id_clean"]
    .map(gene_id_to_symbol)
)

tes_individual["TES_distance_RNA"] = (
    pd.to_numeric(
        tes_individual["TES_distance_RNA"],
        errors="coerce",
    )
)

tes_individual = tes_individual.dropna(
    subset=[
        "sample",
        "gene_symbol",
        "TES_distance_RNA",
    ]
)

tes_reach_vs_snp = gene_reach_chrX.merge(
    tes_individual,
    left_on="gene",
    right_on="gene_symbol",
    how="inner",
)

tes_reach_vs_snp[
    "reachable_by_illumina_150bp"
] = (
    tes_reach_vs_snp["TES_distance_RNA"]
    <= 150
)

tes_reach_vs_snp[
    "reachable_by_nanopore_median"
] = (
    tes_reach_vs_snp["median_reach_bp"]
    >= tes_reach_vs_snp["TES_distance_RNA"]
)

tes_reach_vs_snp[
    "reachable_by_nanopore_p75"
] = (
    tes_reach_vs_snp["p75_reach_bp"]
    >= tes_reach_vs_snp["TES_distance_RNA"]
)

tes_individual_reach_summary = (
    tes_reach_vs_snp
    .groupby("sample")
    .agg(
        n_gene_tests=(
            "gene_symbol",
            "count",
        ),
        illumina_150bp_reachable=(
            "reachable_by_illumina_150bp",
            "sum",
        ),
        nanopore_median_reachable=(
            "reachable_by_nanopore_median",
            "sum",
        ),
        nanopore_p75_reachable=(
            "reachable_by_nanopore_p75",
            "sum",
        ),
    )
    .reset_index()
)

for method in [
    "illumina_150bp",
    "nanopore_median",
    "nanopore_p75",
]:
    tes_individual_reach_summary[
        f"{method}_fraction"
    ] = (
        tes_individual_reach_summary[
            f"{method}_reachable"
        ]
        / tes_individual_reach_summary[
            "n_gene_tests"
        ]
    )


# ============================================================
# Save reachability tables
# ============================================================

tss_reach_vs_snp.to_csv(
    outdir / "TSS_reach_vs_snp_transcript_coordinates.csv.gz",
    index=False,
)

tes_reach_vs_snp.to_csv(
    outdir / "TES_reach_vs_snp_transcript_coordinates.csv.gz",
    index=False,
)

tss_individual_reach_summary.to_csv(
    outdir / "TSS_individual_reach_summary_transcript_coordinates.csv",
    index=False,
)

tes_individual_reach_summary.to_csv(
    outdir / "TES_individual_reach_summary_transcript_coordinates.csv",
    index=False,
)

display(tss_individual_reach_summary.head())
display(tes_individual_reach_summary.head())


# ============================================================
# Plotting function
# ============================================================

def plot_reach_summary(
    summary_df,
    title,
    outfile_prefix,
):
    methods = [
        "illumina_150bp_fraction",
        "nanopore_median_fraction",
        "nanopore_p75_fraction",
    ]

    labels = [
        "150 bp",
        "ONT median\nper gene",
        "ONT P75\nper gene",
    ]

    x = np.arange(len(methods))

    fig, ax = plt.subplots(
        figsize=FIGSIZE_50MM
    )

    # Paired individual lines.
    for _, row in summary_df.iterrows():
        ax.plot(
            x,
            [
                row[method]
                for method in methods
            ],
            color="0.65",
            alpha=0.08,
            linewidth=0.35,
            zorder=1,
        )

    # Boxplots.
    boxplot = ax.boxplot(
        [
            summary_df[method].dropna()
            for method in methods
        ],
        positions=x,
        widths=0.38,
        showfliers=False,
        patch_artist=True,
        zorder=3,
    )

    for patch in boxplot["boxes"]:
        patch.set_facecolor("white")
        patch.set_edgecolor("black")
        patch.set_linewidth(0.7)

    for element in [
        "whiskers",
        "caps",
        "medians",
    ]:
        for item in boxplot[element]:
            item.set_color("black")
            item.set_linewidth(0.7)

    # Individual points.
    rng = np.random.default_rng(1)

    for index, method in enumerate(methods):
        values = (
            summary_df[method]
            .dropna()
            .to_numpy()
        )

        jitter = rng.normal(
            0,
            0.025,
            size=len(values),
        )

        ax.scatter(
            np.full(
                len(values),
                x[index],
            )
            + jitter,
            values,
            s=3.5,
            color="black",
            alpha=0.18,
            linewidths=0,
            zorder=2,
        )

    # Median labels.
    for index, method in enumerate(methods):
        median = summary_df[method].median()

        ax.text(
            x[index],
            median + 0.12,
            f"{median:.2f}",
            ha="center",
            va="bottom",
            fontsize=5.2,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(labels)

    ax.set_ylabel("Fraction reachable")
    ax.set_ylim(0, 0.85)
    ax.set_title(title, pad=3)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.grid(
        axis="y",
        alpha=0.25,
        linewidth=0.4,
    )

    ax.set_axisbelow(True)

    # Keep exported dimensions at exactly 50 mm × 50 mm.
    fig.subplots_adjust(
        left=0.25,
        right=0.98,
        bottom=0.23,
        top=0.88,
    )

    png_path = outdir / f"{outfile_prefix}.png"
    pdf_path = outdir / f"{outfile_prefix}.pdf"
    svg_path = outdir / f"{outfile_prefix}.svg"
    eps_path = outdir / f"{outfile_prefix}.eps"

    fig.savefig(
        png_path,
        dpi=600,
    )

    fig.savefig(pdf_path)
    fig.savefig(svg_path)
    fig.savefig(eps_path)

    print(f"[done] Wrote {png_path}")
    print(f"[done] Wrote {pdf_path}")
    print(f"[done] Wrote {svg_path}")
    print(f"[done] Wrote {eps_path}")

    plt.show()
    plt.close(fig)


# ============================================================
# Produce both plots
# ============================================================

plot_reach_summary(
    tss_individual_reach_summary,
    "TSS SNP coverage",
    "per_individual_TSS_snp_coverage_with_p75_50mm",
)

plot_reach_summary(
    tes_individual_reach_summary,
    "TES SNP coverage",
    "per_individual_TES_snp_coverage_with_p75_50mm",
)